<a href="https://colab.research.google.com/github/PowerRanger18/food-image-recognition/blob/main/food_image_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🍔 Food Image Classification using ResNet

## 📌 Project Overview
This project uses transfer learning with a ResNet model to classify food images from the Food-101 dataset into 101 categories.

## 🎯 Goal
Build a deep learning model that can accurately recognize food images and predict their class.

## 🧠 Model
- Pretrained ResNet (transfer learning)
- Fine-tuned on Food-101 dataset

In [7]:
!pip install torch torchvision

#2. Import library

Imports all required libraries for deep learning, data processing, and model building.

We use PyTorch and Torchvision for building and training a convolutional neural network (CNN) based on ResNet.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
print("step 2 works")

step 2 works


#3. Dataset Loading ( Food101)

Load the Food-101 dataset. It contains 101 different food categories.

Each image is resized and converted into a tensor format so it can be processed by the neural network.

In [2]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_data = datasets.Food101(root='./data', split='train', download=True, transform=transform)
test_data = datasets.Food101(root='./data', split='test', download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

class_names = train_data.classes

print("Model loaded successfully")

100%|██████████| 5.00G/5.00G [04:03<00:00, 20.5MB/s]


Model loaded successfully


# 3. Model Definition (ResNet Transfer Learning)

We use a pretrained ResNet model, which has already learned general image features from ImageNet.

We replace the final layer so the model can classify 101 food categories instead of the original 1000 classes.

In [3]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)

# Freeze feature layers
for param in model.parameters():
    param.requires_grad = False

# Replace classifier
model.fc = nn.Linear(model.fc.in_features, 101)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 205MB/s]


# 4. Training Setup

This section defines:
- Loss function (how the model measures error)
- Optimizer (how the model learns)
- Device (CPU or GPU acceleration)

We use CrossEntropyLoss for multi-class classification.

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# 5. Training Loop

This is where the model learns from the data.

For each epoch, the model:
- Processes training images
- Computes predictions
- Calculates loss
- Updates weights using backpropagation

Over time, the loss should decrease as the model improves.

In [5]:
epochs = 3

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")

Epoch 1, Loss: 2.6077
Epoch 2, Loss: 2.0830
Epoch 3, Loss: 1.9881


# 6. Evaluation

In this step, we test the trained model on unseen data.

The model is evaluated on the test set to measure how well it generalizes.

We calculate accuracy by comparing predictions to true labels.

In [2]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, preds = outputs.max(1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

print("Test Accuracy:", correct / total)

NameError: name 'model' is not defined

# 7. Prediction function

This section demonstrates how to use the trained model on a single image.

The model takes an input image and outputs the predicted food category.



In [7]:
def predict(image):
    model.eval()

    image = image.to(device)

    with torch.no_grad():
        output = model(image.unsqueeze(0))
        pred = output.argmax(1).item()

    return class_names[pred]

#8. Test on Custom Image
This simulates a real-world application of the model.

In [1]:
from PIL import Image
import io
from google.colab import files

uploaded = files.upload()

for filename in uploaded.keys():
    image = Image.open(io.BytesIO(uploaded[filename]))

image = transform(image)
print("Prediction:", predict(image))

Saving pizza.jpg to pizza.jpg


NameError: name 'transform' is not defined

9. Save Model

In [13]:
torch.save(model.state_dict(), "food_resnet.pth")

## ✅ Results
- Model: ResNet18 (transfer learning)
- Dataset: Food-101
- Output: 101-class food classification

## 🚀 Future Improvements
- Fine-tune deeper layers
- Try ResNet50 for higher accuracy
- Add web app interface (Gradio/Flask)